In [1]:
import pandas as pd
import numpy as np
import os 
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import f1_score,confusion_matrix,precision_score,recall_score
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import RandomOverSampler

import matplotlib.pyplot as plt
import seaborn as sns
import shap

from utils.data_utils import correct_col_type,gen_date_col,transform_category_to_counts,min_max_perpatient

In [3]:
# Please change the path with the path of your dataset
DPATH = '../Dataset/'

# Read all tables into data_dict

files = os.listdir(DPATH)
data_dict = {}
summaries = {}
for f in files:
    if 'csv' not in f:
        continue
    print(f)
    fpth = os.path.join(DPATH,f)
    df = pd.read_csv(fpth)

    for col in df.columns:
        df[col] = correct_col_type(df,col)
    if 'date' in df.columns:
        df = df.rename(columns={'date':'timestamp'})
                
    fname = f.split('.')[0]
    data_dict[fname] = df

# physiology data daily

In [4]:
## Aggreate Physiology table in a daily basis by maximum values
phys_df = gen_date_col(data_dict['Physiology'],tcol='timestamp')
phys_df = phys_df.groupby(['patient_id','date','device_type']).agg('max')
phys_df.drop(phys_df.loc[phys_df.sum(axis=1)==0].index,axis=0,inplace=True)
phys_df.reset_index(inplace=True)

phys_df = phys_df.pivot_table(values = 'value', columns='device_type', index=['patient_id','date'])
print(phys_df.columns)
print(phys_df.shape)
phys_df.head()

In [5]:
phys_df.to_csv('../output/data_cleaned/physiology_df_max.csv', index=True)